In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

def extraer_datos_tabla(contenido_html):
    datos = []
    soup = BeautifulSoup(contenido_html, 'html.parser')
    
    # Encontrar la tabla principal
    tabla = soup.find('table', {'id': 'ctl00_ContentPlaceHolder1_GwDetalle'})
    
    if tabla:
        # Buscar todas las filas excepto la cabecera y el pie de página
        filas = tabla.find_all('tr')
        
        for fila in filas:
            # Saltar la fila de cabecera y la de navegación
            if fila.find('th') or fila.find('input', {'type': 'image'}):
                continue
                
            celdas = fila.find_all('td')
            
            if len(celdas) == 5:
                norma = celdas[0].get_text(strip=True)
                
                # Extraer el número y su link
                enlace = celdas[1].find('a')
                if enlace:
                    numero = enlace.get_text(strip=True)
                    # Extraer el link del onclick
                    onclick = enlace.get('href', '')
                    if onclick.startswith('javascript:'):
                        # Extraer la URL del javascript
                        match = re.search(r"'(http[^']+)'", onclick)
                        if match:
                            link = match.group(1)
                        else:
                            link = ''
                    else:
                        link = onclick
                else:
                    numero = celdas[1].get_text(strip=True)
                    link = ''
                
                publicacion = celdas[2].get_text(strip=True)
                descripcion = celdas[3].get_text(strip=True)
                
                datos.append({
                    'Norma': norma,
                    'Número': numero,
                    'Link': link,
                    'Publicación': publicacion,
                    'Descripción': descripcion,
                })
    
    return datos

def obtener_datos_formulario(soup):
    datos_formulario = {}
    
    # Obtener todos los inputs hidden
    inputs_hidden = soup.find_all('input', {'type': 'hidden'})
    
    for input_elem in inputs_hidden:
        nombre = input_elem.get('name')
        valor = input_elem.get('value', '')
        if nombre:
            datos_formulario[nombre] = valor
    
    return datos_formulario

def verificar_boton(soup, nombre_boton):
    """Verifica si el botón está habilitado"""
    boton = soup.find('input', {'name': nombre_boton})
    
    if not boton:
        return False
    
    # Verificar si tiene el atributo disabled
    if boton.has_attr('disabled'):
        return False
    
    # Verificar si la imagen existe y no está en gris
    src = boton.get('src', '')
    if 'gris' in src.lower() or 'disabled' in src.lower():
        return False
    
    return True

url = "https://www.leyes.congreso.gob.pe/LeyNume_1p.aspx?xEstado=2&xTipoNorma=0&xTipoBusqueda=2&xFechaI=01%2f01%2f2020&xFechaF=10%2f10%2f2025&xTexto=&xOrden=0&xNormaI=&xNormaF="

session = requests.Session()
    
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
    'Connection': 'keep-alive',
}

todos_datos = []
pagina_actual = 1

respuesta = session.get(url, headers=headers, timeout=30)

url_actual = respuesta.url

while True:
    soup = BeautifulSoup(respuesta.content, 'lxml')
        
    datos_pagina = extraer_datos_tabla(respuesta.content)
    todos_datos.extend(datos_pagina)

    for dato in datos_pagina:
        print(f"Ley: {dato['Número']} | Descripción: {dato['Descripción']}")
    
    print(f"Página {pagina_actual}: {len(datos_pagina)} registros |" 
          f"Total: {len(todos_datos)}")
        
    boton_siguiente = 'ctl00$ContentPlaceHolder1$GwDetalle$ctl23$ImgBtnSiguiente'
        
    if not verificar_boton(soup, boton_siguiente):
        print("Última página alcanzada")
        break
        
    # Extraer datos
    datos_formulario = obtener_datos_formulario(soup)
    datos_formulario['__EVENTTARGET'] = boton_siguiente
    datos_formulario['__EVENTARGUMENT'] = ''
        
    headers['Referer'] = url_actual
    headers['Content-Type'] = 'application/x-www-form-urlencoded'
        
    pagina_actual += 1

    respuesta = session.post(url_actual, data=datos_formulario, headers=headers, timeout=30)
        
    time.sleep(0.3)

    
df = pd.DataFrame(todos_datos)

print(f"\n{'='*60}")
print(f"Total de registros: {len(df)}")
print(f"Total de páginas: {pagina_actual}")
print(f"{'='*60}\n")

nombre_archivo = 'leyes.csv'
df.to_csv(nombre_archivo, index=False, encoding='utf-8-sig')
print(f"Datos guardados en '{nombre_archivo}'")


Ley: 31010 | Descripción: LEY QUE INCORPORA LA TERCERA DISPOSICIÓN TRANSITORIA DE LA LEY 26859, LEY ORGÁNICA DE ELECCIONES
Ley: 31011 | Descripción: LEY QUE DELEGA EN EL PODER EJECUTIVO LA FACULTAD DE LEGISLAR EN DIVERSAS MATERIAS PARA LA ATENCIÓN DE LA EMERGENCIA SANITARIA PRODUCIDA POR EL COVID-19
Ley: 31012 | Descripción: LEY DE PROTECCIÓN POLICIAL
Ley: 31013 | Descripción: LEY QUE MODIFICA EL ARTÍCULO 34 DE LA LEY 29459, LEY DE LOS PRODUCTOS FARMACÉUTICOS, DISPOSITIVOS MÉDICOS Y PRODUCTOS SANITARIOS
Ley: 31014 | Descripción: LEY QUE REGULA EL USO DE SUSTANCIAS MODELANTES EN TRATAMIENTOS CORPORALES CON FINES ESTÉTICOS Y DEFINE DICHO PROCEDIMIENTO COMO ACTO MÉDICO
Ley: 31015 | Descripción: LEY QUE AUTORIZA LA EJECUCIÓN DE INTERVENCIONES EN INFRAESTRUCTURA SOCIAL BÁSICA, PRODUCTIVA Y NATURAL, MEDIANTE NÚCLEOS EJECUTORES
Ley: 31016 | Descripción: LEY QUE ESTABLECE MEDIDAS PARA DESPLIEGUE DEL CONTROL SIMULTÁNEO DURANTE LA EMERGENCIA SANITARIA POR EL COVID-19
Ley: 31017 | Descripción: LE

In [2]:
df

,Norma,Número,Link,Publicación,Descripción
0,LEY,31010,http://www2.congreso.gob.pe/Sicr/TraDocEstProc...,27/03/2020,LEY QUE INCORPORA LA TERCERA DISPOSICIÓN TRANS...
1,LEY,31011,http://www2.congreso.gob.pe/Sicr/TraDocEstProc...,27/03/2020,LEY QUE DELEGA EN EL PODER EJECUTIVO LA FACULT...
2,LEY,31012,http://www2.congreso.gob.pe/Sicr/TraDocEstProc...,28/03/2020,LEY DE PROTECCIÓN POLICIAL
3,LEY,31013,http://www2.congreso.gob.pe/Sicr/TraDocEstProc...,28/03/2020,LEY QUE MODIFICA EL ARTÍCULO 34 DE LA LEY 2945...
4,LEY,31014,http://www2.congreso.gob.pe/Sicr/TraDocEstProc...,28/03/2020,LEY QUE REGULA EL USO DE SUSTANCIAS MODELANTES...
...,...,...,...,...,...
1449,LEY,32460,http://www2.congreso.gob.pe/Sicr/TraDocEstProc...,02/10/2025,"LEY QUE MODIFICA LA LEY 29230, LEY QUE IMPULSA..."
1450,LEY,32461,http://www2.congreso.gob.pe/Sicr/TraDocEstProc...,02/10/2025,LEY QUE CREA UNIVERSIDADES NACIONALES EN LOS D...
1451,LEY,32462,http://www2.congreso.gob.pe/Sicr/TraDocEstProc...,03/10/2025,"LEY QUE MODIFICA LA LEY 26859, LEY ORGÁNICA DE..."
1452,LEY,32463,http://www2.congreso.gob.pe/Sicr/TraDocEstProc...,04/10/2025,LEY QUE DECLARA DE INTERÉS NACIONAL LA CONSTRU...
